# 📝 Tugas Praktikum — Tempat Kedudukan Akar (*Root Locus*)
### Sistem Kontrol — Teknik Elektro - Universitas Jember

| | |
|---|---|
| **Nama** | *(isi di sini)* |
| **NIM** | *(isi di sini)* |
| **Kelas** | *(isi di sini)* |
| **Tanggal** | *(isi di sini)* |

---

### Petunjuk Pengerjaan
1. **Jalankan sel Setup** di bawah terlebih dahulu.
2. Kerjakan Soal 1–4 secara berurutan.
3. Ganti semua tanda `???` dengan kode atau nilai yang benar.
4. Jawab pertanyaan analisis di sel Markdown (klik dua kali untuk edit).
5. Simpan notebook (`File → Save`) dan kumpulkan file `.ipynb`.

### Bobot Penilaian

| Soal | Topik | Poin |
|------|-------|------|
| 1 | Sistem Orde-2 | 25 |
| 2 | Sistem Orde-3 + K Kritis | 25 |
| 3 | Desain K dari Spesifikasi | 25 |
| 4 *(pengayaan)* | Efek Pole & Zero | 25 |

> Soal 1–3 wajib. Soal 4 menaikkan nilai ke rentang A.


---
## ⚙️ Setup — Jalankan Sel Ini Pertama Kali
Sel di bawah menginstal library dan mendefinisikan semua fungsi bantu.
**Jangan diubah.**


In [ ]:
# ── Setup: instalasi dan definisi fungsi (JANGAN DIUBAH) ─────────────
%pip install -q control

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import control
from control import tf, feedback, step_response
import warnings; warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11,
                     'axes.labelsize': 10, 'figure.dpi': 100})
C1, C2, C3, C4 = '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'

# ── Utilitas ──────────────────────────────────────────────────────────
def cetak_info_sistem(sys_tf, nama="Sistem"):
    poles = control.poles(sys_tf); zeros = control.zeros(sys_tf)
    n, m = len(poles), len(zeros)
    print(f"\n{'='*52}\n  INFO: {nama}\n{'='*52}")
    print(f"  Poles  : {np.round(poles,4)}")
    print(f"  Zeros  : {np.round(zeros,4) if m>0 else 'Tidak ada'}")
    print(f"  Orde   : {n}   |   Asimtot: {n-m}")
    if n > m:
        sudut = [(2*k+1)*180/(n-m) for k in range(n-m)]
        centroid = (sum(poles.real)-sum(zeros.real))/(n-m)
        print(f"  Sudut asimtot : {[round(s,1) for s in sudut]} °")
        print(f"  Centroid      : {round(centroid.real,4)}")
    print('='*52)

def hitung_parameter_respons(t, y, nama="", tampilkan=True):
    y_ss = y[-1]
    i10 = np.where(y >= 0.1*y_ss)[0]; i90 = np.where(y >= 0.9*y_ss)[0]
    rt = float(t[i90[0]]-t[i10[0]]) if (len(i10) and len(i90)) else None
    ip = np.argmax(y)
    os_pct = max(0, (y[ip]-y_ss)/y_ss*100) if y_ss else 0
    luar = np.where((y > y_ss*1.02) | (y < y_ss*0.98))[0]
    ts = float(t[luar[-1]]) if len(luar) else float(t[0])
    par = {'rise_time': round(rt,4) if rt else None,
           'peak_time': round(float(t[ip]),4),
           'overshoot': round(os_pct,2),
           'settling_time': round(ts,4),
           'steady_state': round(float(y_ss),4),
           'ess': round(abs(1.0-float(y_ss)),4)}
    if tampilkan and nama:
        print(f"\n  Parameter Transien [{nama}]")
        for k, v in par.items():
            print(f"    {k:<16}: {v}")
    return par

def plot_root_locus(sys_tf, judul="Root Locus", K_tandai=None, ax=None):
    standalone = ax is None
    if standalone: fig, ax = plt.subplots(figsize=(8, 6))
    rlist, _ = control.root_locus(sys_tf, plot=False)
    for i in range(rlist.shape[1]):
        ax.plot(rlist[:,i].real,  rlist[:,i].imag,  color=C1, lw=2, alpha=0.85)
        ax.plot(rlist[:,i].real, -rlist[:,i].imag, color=C1, lw=2, alpha=0.85)
    p = control.poles(sys_tf); z = control.zeros(sys_tf)
    ax.plot(p.real, p.imag, 'x', color=C4, ms=12, mew=2.5, label='Poles', zorder=5)
    if len(z): ax.plot(z.real, z.imag, 'o', color=C3, ms=10, mew=2.5, mfc='none',
                       label='Zeros', zorder=5)
    for pi in p: ax.annotate(f'  {pi:.2f}', xy=(pi.real,pi.imag), fontsize=8, color=C4)
    for zi in z: ax.annotate(f'  {zi:.2f}', xy=(zi.real,zi.imag), fontsize=8, color=C3)
    if K_tandai:
        for Kv in K_tandai:
            cp = control.poles(feedback(Kv*sys_tf, 1))
            ax.plot(cp.real, cp.imag, 's', color=C2, ms=9, zorder=6, label=f'K={Kv}')
    ax.axhline(0, color='gray', lw=0.8, ls='--', alpha=0.5)
    ax.axvline(0, color=C4, lw=1.5, ls='-.', alpha=0.4, label='Batas stabil')
    ax.set_xlabel('Re(s)'); ax.set_ylabel('Im(s)')
    ax.set_title(judul, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right'); ax.grid(True, alpha=0.3, ls=':')
    if standalone: plt.tight_layout(); plt.show(); return fig, ax

def plot_respons_step_variasi_K(sys_tf, nilai_K, t_max=20, judul="Respons Step"):
    t = np.linspace(0, t_max, 2000)
    fig, ax = plt.subplots(figsize=(10, 4.5))
    colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(nilai_K)))
    for K, warna in zip(nilai_K, colors):
        t_o, y_o = step_response(feedback(K*sys_tf, 1), T=t)
        ax.plot(t_o, y_o, color=warna, lw=2, label=f'K = {K}')
    ax.axhline(1,    color='gray', lw=1.2, ls='--', alpha=0.7, label='r(t)=1')
    ax.axhline(1.02, color='red',  lw=0.8, ls=':',  alpha=0.5)
    ax.axhline(0.98, color='red',  lw=0.8, ls=':',  alpha=0.5, label='±2%')
    ax.set_xlabel('Waktu (s)'); ax.set_ylabel('y(t)')
    ax.set_title(judul, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8.5, loc='upper right'); ax.grid(True, alpha=0.3, ls=':')
    ax.set_xlim(0, t_max); plt.tight_layout(); plt.show(); return fig, ax

def plot_dashboard_lengkap(sys_tf, K_desain, nama_sistem="Sistem",
                            t_max=20, nilai_K_banding=None):
    fig = plt.figure(figsize=(15, 9))
    fig.suptitle(f'Analisis Lengkap: {nama_sistem}', fontsize=13, fontweight='bold')
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.4)
    ax_rl=fig.add_subplot(gs[0,0]); ax_st=fig.add_subplot(gs[0,1:])
    ax_mg=fig.add_subplot(gs[1,0]); ax_ph=fig.add_subplot(gs[1,1]); ax_tb=fig.add_subplot(gs[1,2])
    rlist,_=control.root_locus(sys_tf, plot=False)
    for i in range(rlist.shape[1]):
        ax_rl.plot(rlist[:,i].real, rlist[:,i].imag, color=C1,lw=1.8)
        ax_rl.plot(rlist[:,i].real,-rlist[:,i].imag, color=C1,lw=1.8)
    poles=control.poles(sys_tf); zeros=control.zeros(sys_tf)
    ax_rl.plot(poles.real,poles.imag,'x',color=C4,ms=11,mew=2.5,label='Poles')
    if len(zeros): ax_rl.plot(zeros.real,zeros.imag,'o',color=C3,ms=9,mew=2.5,mfc='none',label='Zeros')
    cp=control.poles(feedback(K_desain*sys_tf,1))
    ax_rl.plot(cp.real,cp.imag,'s',color=C2,ms=9,label=f'K={K_desain}',zorder=5)
    ax_rl.axvline(0,color='red',lw=1,ls='-.',alpha=0.4); ax_rl.axhline(0,color='gray',lw=0.7,ls='--',alpha=0.4)
    ax_rl.set_title('Root Locus',fontsize=10,fontweight='bold'); ax_rl.set_xlabel('Re(s)'); ax_rl.set_ylabel('Im(s)')
    ax_rl.legend(fontsize=7); ax_rl.grid(True,alpha=0.3,ls=':')
    t=np.linspace(0,t_max,2000); semua_K=[K_desain]+(nilai_K_banding or [])
    colors_k=plt.cm.tab10(np.linspace(0,0.6,len(semua_K)))
    for Kv,warna in zip(semua_K,colors_k):
        t_o,y_o=step_response(feedback(Kv*sys_tf,1),T=t)
        lw=2.5 if Kv==K_desain else 1.5; ls='-' if Kv==K_desain else '--'
        ax_st.plot(t_o,y_o,color=warna,lw=lw,ls=ls,label=f'K={Kv}'+(' ←' if Kv==K_desain else ''))
    ax_st.axhline(1,color='gray',lw=1.2,ls='--',alpha=0.7); ax_st.axhline(1.02,color='red',lw=0.8,ls=':',alpha=0.4)
    ax_st.axhline(0.98,color='red',lw=0.8,ls=':',alpha=0.4)
    ax_st.set_title('Respons Step',fontsize=10,fontweight='bold'); ax_st.set_xlabel('Waktu (s)'); ax_st.set_ylabel('y(t)')
    ax_st.legend(fontsize=7.5,loc='upper right'); ax_st.grid(True,alpha=0.3,ls=':'); ax_st.set_xlim(0,t_max)
    omega=np.logspace(-2,2,500)
    try:
        mag,phase,om=control.bode(K_desain*sys_tf,omega,plot=False,dB=True)
        ax_mg.semilogx(om,20*np.log10(mag+1e-12),color=C1,lw=2)
        ax_mg.axhline(0,color='gray',lw=0.8,ls='--',alpha=0.6)
        ax_mg.set_title('Bode Magnitudo',fontsize=10,fontweight='bold'); ax_mg.set_xlabel('ω'); ax_mg.set_ylabel('dB')
        ax_mg.grid(True,which='both',alpha=0.3,ls=':')
        ax_ph.semilogx(om,np.degrees(phase),color=C2,lw=2)
        ax_ph.axhline(-180,color='red',lw=0.8,ls='--',alpha=0.6,label='-180°')
        ax_ph.set_title('Bode Fase',fontsize=10,fontweight='bold'); ax_ph.set_xlabel('ω'); ax_ph.set_ylabel('°')
        ax_ph.legend(fontsize=8); ax_ph.grid(True,which='both',alpha=0.3,ls=':')
    except Exception: pass
    t_o,y_o=step_response(feedback(K_desain*sys_tf,1),T=t)
    par=hitung_parameter_respons(t_o,y_o,tampilkan=False)
    try:
        gm,pm,_,_=control.margin(K_desain*sys_tf)
        gm_s=f"{20*np.log10(gm):.1f}" if (gm and gm>0) else '∞'; pm_s=f"{pm:.1f}" if pm else '—'
    except Exception: gm_s,pm_s='—','—'
    rows=[['K desain',str(K_desain),'—'],['Rise time',str(par['rise_time']),'s'],
          ['% Overshoot',str(par['overshoot']),'%'],['Settling time',str(par['settling_time']),'s'],
          ['Error SS',str(par['ess']),'—'],['GM',gm_s,'dB'],['PM',pm_s,'°']]
    ax_tb.axis('off')
    tbl=ax_tb.table(cellText=rows,colLabels=['Parameter','Nilai','Satuan'],loc='center',cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.1,1.6)
    for j in range(3): tbl[(0,j)].set_facecolor('#2c3e50'); tbl[(0,j)].set_text_props(color='white',fontweight='bold')
    ax_tb.set_title('Ringkasan',fontsize=10,fontweight='bold')
    plt.tight_layout(rect=[0,0,1,0.96]); plt.show(); return fig

print("✓ Setup selesai. Lanjutkan ke Soal 1.")


---
## Soal 1 — Sistem Orde-2 *(25 poin)*

### Latar Belakang

Sebuah sistem kendali posisi motor DC memiliki fungsi alih loop-terbuka:

$$G(s) = \frac{K}{s(s+4)}$$

dengan umpan balik satuan $H(s) = 1$.

### Tujuan
- Menggambar dan menganalisis root locus sistem orde-2.
- Memahami hubungan posisi poles dengan respons transien.
- Menentukan nilai K dari spesifikasi.


### 1a — Definisi Fungsi Alih *(3 poin)*

In [ ]:
# G(s) = 1 / [s(s+4)] = 1 / (s^2 + 4s)
# tf(pembilang, penyebut) — koefisien dari pangkat TERTINGGI ke terendah

num_G1 = [???]            # ← pembilang
den_G1 = [???, ???, ???]  # ← penyebut: s^2 + 4s + 0

G1 = tf(num_G1, den_G1)
cetak_info_sistem(G1, "Soal 1 — G(s) = 1/[s(s+4)]")


### 1b — Perhitungan Manual *(5 poin)*

Lengkapi tabel berikut berdasarkan teori root locus:

| Besaran | Nilai |
|---------|-------|
| Poles G(s) | s₁ = ___ , s₂ = ___ |
| Zeros G(s) | *(ada/tidak ada)* |
| Jumlah asimtot (n−m) | ___ |
| Sudut asimtot | ___ ° |
| Centroid asimtot σ_a | ___ |
| Break-away point (s) | ___ |
| K saat break-away | ___ |

**Petunjuk break-away:**
$K = -\frac{1}{G(s)}$ di sepanjang sumbu real, lalu cari $\frac{dK}{ds} = 0$.


### 1c — Plot Root Locus *(7 poin)*
Tandai K = 1, 4, dan 8 pada plot.

In [ ]:
plot_root_locus(
    G1,
    judul="Root Locus — G(s) = 1/[s(s+4)]",
    K_tandai=[???, ???, ???]   # ← isi tiga nilai K
)


### 1d — Respons Step Variasi K *(5 poin)*
Plot respons step untuk K = 1, 4, 8, **16**.

In [ ]:
plot_respons_step_variasi_K(
    G1,
    nilai_K=[1, 4, 8, ???],    # ← tambahkan K = 16
    t_max=10,
    judul="Respons Step — G(s) = 1/[s(s+4)]"
)


### 1e — Parameter Transien untuk K = 4 *(5 poin)*

In [ ]:
K_1e = ???   # ← nilai K = 4

cl_1e = feedback(K_1e * G1, 1)
t_1e, y_1e = step_response(cl_1e, T=np.linspace(0, 10, 2000))
par_1e = hitung_parameter_respons(t_1e, y_1e, nama=f"Soal 1e, K={K_1e}")


### Pertanyaan Analisis 1 *(jawab dengan mengklik dua kali sel ini)*

**1.** Berapa nilai % Overshoot untuk K = 4?
➡ *Jawaban:*

**2.** Apakah sistem ini stabil untuk **semua** nilai K > 0? Jelaskan mengapa!
➡ *Jawaban:*

**3.** Bagaimana pengaruh menaikkan K terhadap *settling time*?
➡ *Jawaban:*


---
## Soal 2 — Sistem Orde-3 & K Kritis *(25 poin)*

### Latar Belakang

Sistem kendali level cairan pada reaktor kimia:

$$G(s) = \frac{K}{s(s+2)(s+5)}$$

### Tujuan
- Menganalisis root locus sistem orde-3.
- Menentukan K kritis menggunakan kriteria **Routh-Hurwitz**.
- Memverifikasi K kritis dengan simulasi.


### 2a — Definisi G(s) *(3 poin)*
Kalikan dulu: $s(s+2)(s+5) = s^3 + 7s^2 + 10s$

In [ ]:
# G(s) = 1 / (s^3 + 7s^2 + 10s)
num_G2 = [???]
den_G2 = [???, ???, ???, ???]   # ← 4 koefisien

G2 = tf(num_G2, den_G2)
cetak_info_sistem(G2, "Soal 2 — G(s) = 1/[s(s+2)(s+5)]")


### 2b — Routh-Hurwitz Manual *(7 poin)*

Persamaan karakteristik loop-tertutup: $1 + KG(s) = 0$
→ $s^3 + 7s^2 + 10s + K = 0$

**Tabel Routh** (lengkapi sel yang kosong):

| | Kolom 1 | Kolom 2 |
|--|:---:|:---:|
| $s^3$ | 1 | 10 |
| $s^2$ | 7 | K |
| $s^1$ | ??? | 0 |
| $s^0$ | K | — |

**Syarat stabil** (semua elemen kolom 1 > 0):
Dari baris $s^1$: ??? > 0 → K < ???
Dari baris $s^0$: K > 0

∴ **K kritis = ???**


In [ ]:
# Masukkan nilai K kritis hasil perhitungan Routh-Hurwitz
K_kritis_routh = ???   # ← isi nilai K kritis
print(f"K kritis (Routh-Hurwitz) = {K_kritis_routh}")


### 2c — Verifikasi Simulasi *(5 poin)*
Bandingkan respons step untuk K di bawah, tepat di, dan di atas K kritis.

In [ ]:
K_bawah = K_kritis_routh * 0.5
K_tepat = K_kritis_routh
K_atas  = K_kritis_routh * 1.5

plot_respons_step_variasi_K(
    G2,
    nilai_K=[K_bawah, K_tepat, K_atas],
    t_max=30,
    judul=f"Verifikasi K Kritis ≈ {K_kritis_routh}"
)


### 2d — Dashboard Lengkap *(10 poin)*
Gunakan K desain = 50% × K kritis (margin keamanan).

In [ ]:
K_desain_2d = round(K_kritis_routh * 0.5, 2)
print(f"K desain = {K_desain_2d}")

plot_dashboard_lengkap(
    G2,
    K_desain=K_desain_2d,
    nama_sistem="Soal 2 — G(s)=1/[s(s+2)(s+5)]",
    t_max=25,
    nilai_K_banding=[1, K_kritis_routh]
)


### Pertanyaan Analisis 2 *(klik dua kali untuk edit)*

**1.** Bandingkan K kritis dari Routh-Hurwitz vs simulasi. Apakah konsisten?
➡ *Jawaban:*

**2.** Mengapa sistem orde-3 bisa tidak stabil, sementara sistem orde-2 (Soal 1) selalu stabil?
*(Petunjuk: perhatikan arah asimtot)*
➡ *Jawaban:*

**3.** Apa trade-off memilih K kecil vs K mendekati K_kritis?
➡ *Jawaban:*


---
## Soal 3 — Desain K Berdasarkan Spesifikasi *(25 poin)*

### Latar Belakang

Sistem kendali kecepatan turbin angin: $G(s) = \dfrac{K}{s(s+3)}$

**Spesifikasi desain:**
- % Overshoot ≤ **15%**
- Settling time ≤ **5 detik** (kriteria ±2%)
- Error steady-state = **0** (sistem tipe 1)

### Tujuan
- Mengonversi spesifikasi ke daerah bidang-s.
- Menentukan K optimal dengan pendekatan *scanning*.


### 3a — Konversi Spesifikasi ke Bidang-s *(7 poin)*

**%OS ≤ 15% → minimum damping ratio ζ:**

$$\zeta = \frac{-\ln(\text{OS}/100)}{\sqrt{\pi^2 + \ln^2(\text{OS}/100)}}$$

**Settling time ≤ 5 s → minimum σ:**

$$\sigma_{\min} = \frac{4}{t_{s,\max}} \quad \Rightarrow \quad \text{Re}(\text{poles}) \leq -\sigma_{\min}$$


In [ ]:
OS_maks = 15.0   # %
ts_maks = 5.0    # detik

# Hitung zeta minimum
zeta_min = ???   # ← isi rumus: -np.log(OS_maks/100) / np.sqrt(np.pi**2 + np.log(OS_maks/100)**2)

# Hitung sigma minimum
sigma_min = ???  # ← isi rumus: 4 / ts_maks

print(f"ζ minimum    = {zeta_min:.4f}")
print(f"σ minimum    = {sigma_min:.4f}  → Re(poles) ≤ -{sigma_min:.4f}")


### 3b — Plot Root Locus + Daerah Spesifikasi *(8 poin)*

In [ ]:
G3 = tf([1], [1, 3, 0])   # G(s) = 1/[s(s+3)]
cetak_info_sistem(G3, "Soal 3 — G(s) = 1/[s(s+3)]")

fig3b, ax3b = plt.subplots(figsize=(9, 7))

# Root locus
rlist, _ = control.root_locus(G3, plot=False)
for i in range(rlist.shape[1]):
    ax3b.plot(rlist[:,i].real,  rlist[:,i].imag,  color=C1, lw=2.2)
    ax3b.plot(rlist[:,i].real, -rlist[:,i].imag, color=C1, lw=2.2)
poles_G3 = control.poles(G3)
ax3b.plot(poles_G3.real, poles_G3.imag, 'x', color=C4, ms=12, mew=2.5, label='Poles')

# Garis batas settling time
ax3b.axvline(-sigma_min, color='green', lw=2, ls='--',
             label=f'ts ≤ {ts_maks}s (Re ≤ -{sigma_min:.2f})')

# Garis batas %OS (diagonal dari origin)
x_diag = np.linspace(-8, 0, 300)
theta  = np.arccos(zeta_min)
ax3b.plot(x_diag,  np.tan(theta)*abs(x_diag), color='orange', lw=2, ls='--',
          label=f'OS ≤ {OS_maks}% (ζ ≥ {zeta_min:.2f})')
ax3b.plot(x_diag, -np.tan(theta)*abs(x_diag), color='orange', lw=2, ls='--')

# Arsiran daerah memenuhi spesifikasi
x_ok = x_diag[x_diag <= -sigma_min]
if len(x_ok):
    y_ok = np.tan(theta) * abs(x_ok)
    ax3b.fill_between(x_ok, -y_ok, y_ok, alpha=0.12, color='green',
                      label='✓ Daerah specs terpenuhi')

ax3b.axhline(0, color='gray', lw=0.7, ls='--', alpha=0.5)
ax3b.axvline(0, color=C4, lw=1.2, ls='-.', alpha=0.4, label='Batas stabil')
ax3b.set_xlim(-8, 2); ax3b.set_ylim(-6, 6)
ax3b.set_xlabel('Re(s)  σ', fontsize=11); ax3b.set_ylabel('Im(s)  jω', fontsize=11)
ax3b.set_title('Root Locus + Daerah Spesifikasi\nG(s) = 1/[s(s+3)]',
               fontsize=12, fontweight='bold')
ax3b.legend(loc='upper right', fontsize=8.5)
ax3b.grid(True, alpha=0.3, ls=':')
plt.tight_layout(); plt.show()


### 3c — Pencarian K Optimal dengan Scanning *(5 poin)*

In [ ]:
K_scan3 = np.linspace(0.1, 10, 500)
t_sim3  = np.linspace(0, 25, 3000)

hasil = []
for K in K_scan3:
    cl = feedback(K * G3, 1)
    if any(p.real >= 0 for p in control.poles(cl)):
        continue
    t_o, y_o = step_response(cl, T=t_sim3)
    par = hitung_parameter_respons(t_o, y_o, tampilkan=False)
    hasil.append({'K': K, 'OS': par['overshoot'], 'ts': par['settling_time']})

# Filter yang memenuhi semua spesifikasi
valid = [r for r in hasil if r['OS'] <= OS_maks and r['ts'] <= ts_maks]

if valid:
    K_opt = min(valid, key=lambda r: r['ts'])
    print(f"✓ K optimal = {K_opt['K']:.3f}")
    print(f"  % OS      = {K_opt['OS']:.2f}%   (syarat ≤ {OS_maks}%)")
    print(f"  ts        = {K_opt['ts']:.3f} s  (syarat ≤ {ts_maks} s)")
else:
    print("✗ Tidak ada K yang memenuhi semua spesifikasi. Periksa zeta_min dan sigma_min.")
    K_opt = {'K': 2.0}


### 3d — Verifikasi K Optimal *(5 poin)*

In [ ]:
plot_dashboard_lengkap(
    G3,
    K_desain=round(K_opt['K'], 3),
    nama_sistem="Soal 3 — Turbin Angin (K Optimal)",
    t_max=20,
    nilai_K_banding=[1.0, 5.0]
)


### Pertanyaan Analisis 3 *(klik dua kali untuk edit)*

**1.** Berapa nilai K optimal yang Anda temukan?
➡ *K_optimal = ___*

**2.** Apakah ada konflik antara spesifikasi OS dan settling time?
(Misal: K yang memperkecil OS justru memperbesar ts?)
➡ *Jawaban:*

**3.** Mengapa error steady-state sistem tipe-1 dengan input step = 0?
*(Petunjuk: ingat teorema nilai akhir dan konstanta posisi Kp)*
➡ *Jawaban:*


---
## Soal 4 — Pengayaan: Efek Penambahan Pole & Zero *(25 poin)*

### Latar Belakang

Dalam perancangan kompensator (lead/lag), kita menambahkan pole atau zero.
Soal ini menyelidiki pengaruhnya terhadap bentuk root locus.

**Sistem dasar:** $G_0(s) = \dfrac{K}{s(s+2)}$

| Label | Fungsi Alih | Modifikasi |
|-------|------------|------------|
| A | $G_0$ | Baseline |
| B | $\frac{K(s+1)}{s(s+2)}$ | Tambah zero di $s=-1$ |
| C | $\frac{K(s+3)}{s(s+2)}$ | Tambah zero di $s=-3$ |
| D | $\frac{K}{s(s+2)(s+10)}$ | Tambah pole di $s=-10$ (jauh) |
| E | $\frac{K}{s(s+2)(s+1)}$ | Tambah pole di $s=-1$ (dekat) |


### 4a — Definisi Semua Varian *(5 poin)*

In [ ]:
G0 = tf([1],    [1, 2, 0])         # baseline
GA = tf([???, ???], [???, ???, ???])  # zero s=-1 → num=[1,1]
GB = tf([???, ???], [???, ???, ???])  # zero s=-3 → num=[1,3]
GC = tf([1],    [???, ???, ???, ???]) # pole s=-10 → den: s(s+2)(s+10)
GD = tf([1],    [???, ???, ???, ???]) # pole s=-1  → den: s(s+2)(s+1)

varian = {
    'A — Baseline': G0,
    'B — Zero s=-1': GA,
    'C — Zero s=-3': GB,
    'D — Pole s=-10 (jauh)': GC,
    'E — Pole s=-1 (dekat)': GD,
}
print("Varian didefinisikan:", list(varian.keys()))


### 4b — Root Locus Semua Varian *(10 poin)*

In [ ]:
fig4b, axes = plt.subplots(2, 3, figsize=(16, 10))
fig4b.suptitle('Efek Penambahan Pole/Zero pada Root Locus',
               fontsize=13, fontweight='bold')
palette = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd']

for idx, (label, G_var) in enumerate(varian.items()):
    ax = axes.flatten()[idx]
    try:
        rlist, _ = control.root_locus(G_var, plot=False)
        for i in range(rlist.shape[1]):
            ax.plot(rlist[:,i].real,  rlist[:,i].imag,  color=palette[idx], lw=2)
            ax.plot(rlist[:,i].real, -rlist[:,i].imag, color=palette[idx], lw=2)
        p = control.poles(G_var); z = control.zeros(G_var)
        ax.plot(p.real, p.imag, 'x', color='red',   ms=11, mew=2.5, label='Poles')
        if len(z): ax.plot(z.real, z.imag, 'o', color='green', ms=9, mew=2.5,
                           mfc='none', label='Zeros')
    except Exception as e:
        ax.text(0.5, 0.5, f'Error:\n{e}', ha='center', va='center',
                transform=ax.transAxes, fontsize=8)
    ax.axvline(0, color='red', lw=1, ls='-.', alpha=0.4)
    ax.axhline(0, color='gray', lw=0.6, ls='--', alpha=0.4)
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.set_xlabel('Re(s)', fontsize=8); ax.set_ylabel('Im(s)', fontsize=8)
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3, ls=':'); ax.set_xlim(-12, 2)

axes.flatten()[5].axis('off')
plt.tight_layout(); plt.show()


### 4c — Perbandingan Respons Step (K = 2) *(5 poin)*

In [ ]:
K_banding = 2.0
t_sim4 = np.linspace(0, 20, 2000)
fig4c, ax4c = plt.subplots(figsize=(11, 5))

for idx, (label, G_var) in enumerate(varian.items()):
    try:
        cl_v = feedback(K_banding * G_var, 1)
        if any(p.real >= 0 for p in control.poles(cl_v)):
            print(f"✗ {label}: TIDAK STABIL untuk K={K_banding}")
            continue
        t_o, y_o = step_response(cl_v, T=t_sim4)
        lw = 2.8 if idx == 0 else 1.8
        ax4c.plot(t_o, y_o, color=palette[idx], lw=lw, label=label)
    except Exception as e:
        print(f"✗ {label}: {e}")

ax4c.axhline(1,    color='gray', lw=1.2, ls='--', alpha=0.7, label='r(t)=1')
ax4c.axhline(1.02, color='red',  lw=0.7, ls=':',  alpha=0.5)
ax4c.axhline(0.98, color='red',  lw=0.7, ls=':',  alpha=0.5, label='±2%')
ax4c.set_xlabel('Waktu (s)', fontsize=11); ax4c.set_ylabel('y(t)', fontsize=11)
ax4c.set_title(f'Perbandingan Respons Step — K = {K_banding}',
               fontsize=12, fontweight='bold')
ax4c.legend(loc='upper right', fontsize=8); ax4c.grid(True, alpha=0.3, ls=':')
ax4c.set_xlim(0, 20); plt.tight_layout(); plt.show()


### Pertanyaan Analisis 4 *(klik dua kali untuk edit)*

**1.** Apa perbedaan efek zero di $s=-1$ (dekat origin) vs $s=-3$ (jauh dari origin)?
➡ *Jawaban:*

**2.** Bandingkan penambahan pole jauh ($s=-10$) vs dekat ($s=-1$) terhadap sistem baseline.
Mana yang lebih mendekati sistem asli? Mengapa?
➡ *Jawaban:*

**3.** Dalam desain kompensator *lead*, mengapa zero biasanya ditempatkan lebih dekat ke origin daripada pole-nya?
➡ *Jawaban:*

**4.** Sebutkan dan jelaskan minimal **5 aturan konstruksi** Root Locus yang Anda gunakan dalam tugas ini:
➡ 1. ___
➡ 2. ___
➡ 3. ___
➡ 4. ___
➡ 5. ___


---
## ✅ Pengecekan Akhir

Sebelum mengumpulkan, pastikan semua sel sudah dijalankan dan tidak ada error merah.


In [ ]:
print("=" * 55)
print("  CEK AKHIR TUGAS ROOT LOCUS")
print("=" * 55)

cek = {
    "Setup berhasil": True,
    "Soal 1: G1 terdefinisi": 'G1' in dir(),
    "Soal 2: G2 terdefinisi": 'G2' in dir(),
    "Soal 2: K_kritis_routh diisi": 'K_kritis_routh' in dir() and K_kritis_routh != 0,
    "Soal 3: G3 terdefinisi": 'G3' in dir(),
    "Soal 3: zeta_min dihitung": 'zeta_min' in dir(),
    "Soal 3: K_opt ditemukan": 'K_opt' in dir(),
    "Soal 4: Semua varian didefinisi": all(v in dir() for v in ['GA','GB','GC','GD']),
}
for item, status in cek.items():
    simbol = "✓" if status else "✗ BELUM SELESAI"
    print(f"  {simbol}  {item}")

print("=" * 55)
print("Simpan notebook: File → Save (Ctrl+S)")


---
### Kesimpulan

*(Tulis kesimpulan Anda di sini — minimal 100 kata. Apa yang Anda pelajari dari topik Root Locus? Bagaimana nilai K mempengaruhi kestabilan dan kualitas respons sistem?)*

---
*Notebook ini dibuat untuk keperluan Kuliah Sistem Kontrol.*
